# Submissão 1 - Modelo PyTorch

In [1]:
import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn

# ==========================================
# 1. DEFINIR A REDE (Tem de ser igual ao treino!)
# ==========================================
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# ==========================================
# 2. CARREGAR FICHEIROS, VECTORIZER E MODELO
# ==========================================
print("A carregar modelo e dados...")

device = torch.device('cpu') 

df_subm = pd.read_csv('subm1.csv', sep=';')

# Carregar o Vectorizer (que tem as 5000 palavras e bigrams guardados)
with open('vectorizer_pytorch.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

# Saber o tamanho da entrada com base no vectorizer
input_dim = len(vectorizer.get_feature_names_out())

modelo = MLP(input_dim=input_dim, num_classes=5).to(device)

# Carregar os pesos
modelo.load_state_dict(torch.load('pesos_pytorch.pth', map_location=device))
modelo.eval() # Colocar em modo de avaliação (desliga o Dropout)

print("Modelo PyTorch carregado com sucesso!")

# ==========================================
# 3. TRANSFORMAR O TEXTO EM NÚMEROS (Tensores)
# ==========================================
print("A vetorizar os textos do subm1.csv...")

# O Scikit-learn faz a vetorização toda automaticamente de uma só vez
X_subm_np = vectorizer.transform(df_subm['Text']).toarray()

# Converter para o formato que o PyTorch percebe (Tensors)
X_subm_tensor = torch.tensor(X_subm_np, dtype=torch.float32).to(device)

# ==========================================
# 4. FAZER A CLASSIFICAÇÃO
# ==========================================
print("A classificar (IAs vs Humano)...")

# Fazer a previsão sem calcular gradientes (mais rápido e economiza memória)
with torch.no_grad():
    previsoes_prob = modelo(X_subm_tensor)
    
# Obter o índice da classe com maior probabilidade e converter de volta para NumPy
classes_previstas_idx = torch.argmax(previsoes_prob, dim=1).numpy()

# Dicionário de conversão
idx_to_label = {0: 'Human', 1: 'OpenAI', 2: 'Google', 3: 'Meta', 4: 'Anthropic'}

# Adicionar ao DataFrame
df_subm['Labels'] = [idx_to_label[idx] for idx in classes_previstas_idx]

# ==========================================
# 5. EXPORTAR O FICHEIRO FINAL
# ========================================== 

# Nome com "B" para indicar a submissão PyTorch
nome_ficheiro_saida = "subm1-g7-MEI-B.csv"

# Guardar o CSV
df_subm.to_csv(nome_ficheiro_saida, sep=';', index=False)

print(f"Concluído! O ficheiro '{nome_ficheiro_saida}' foi gerado.")

# Visualizar as primeiras linhas
df_subm.head()

A carregar modelo e dados...
Modelo PyTorch carregado com sucesso!
A vetorizar os textos do subm1.csv...
A classificar (IAs vs Humano)...
Concluído! O ficheiro 'subm1-g7-MEI-B.csv' foi gerado.


,ID,Text,Labels
0,D2-1,A covalent bond is a chemical bond that involv...,Human
1,D2-2,A covalent bond forms when two atoms share one...,Human
2,D2-3,A covalent bond is a type of chemical bond whe...,Anthropic
3,D2-4,A covalent bond is a chemical bond that involv...,OpenAI
4,D2-5,Driven by exciting developments in the field o...,Anthropic
